# FusionClip Google Colab GPU Worker (v1 Prototype)

This notebook turns your Google Colab instance (T4 or L4 GPU) into a remote execution worker for FusionClip.

### Scope (per Ticket #88 Decision):
- **SDXL Image Generation** (`image_generation`)
- **Real-ESRGAN / Upscale** (`upscale`, planned/next)
- Direct outbound WebSocket/HTTP connection to FusionClip server
- True per-step progress reporting and real artifact uploads to FusionClip storage


In [ ]:
# Step 1: Check GPU environment and install dependencies
!nvidia-smi

!pip install -q diffusers transformers accelerate invisible-watermark safetensors
!pip install -q websocket-client requests psutil gputil pillow pyngrok


In [ ]:
# Step 2: Configure FusionClip Connection Settings
# Enter your FusionClip server URL (or ngrok/tunnel public URL) and secret token
FUSIONCLIP_SERVER_URL = "http://YOUR_SERVER_URL:8000"
FUSIONCLIP_SECRET_KEY = "fc_secret_test_key_12345"

print(f"Target Server: {FUSIONCLIP_SERVER_URL}")


In [ ]:
# Step 3: Load Generative Pipeline (SDXL Turbo for fast inference on T4)
import torch
from diffusers import AutoPipelineForText2Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing SDXL on device: {device}")

sdxl_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None
).to(device)
print("SDXL pipeline loaded successfully.")


In [ ]:
# Step 4: Standalone Smoke Test Cell
# Test model generation locally on the Colab GPU before connecting to server
prompt = "A cinematic shot of a futuristic drone hovering in a neon city, 8k resolution"
print(f"Running smoke test generation: {prompt}")

result_img = sdxl_pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
result_img.save("/tmp/smoke_test_output.png")
print("Saved smoke test output to /tmp/smoke_test_output.png")

# Display result in notebook
from IPython.display import display
display(result_img)


In [ ]:
# Step 5: Colab Worker with Handler Registry, Thread Safety, and HTTP Fallback
import os
import sys
import time
import json
import threading
import requests
import psutil
import websocket

class ColabWorker:
    def __init__(self, server_url, token):
        self.server_url = server_url.rstrip("/")
        self.token = token
        self.is_running = True
        self.ws_connected = False
        self.ws = None
        self._gpu_lock = threading.Lock()
        self._ws_lock = threading.Lock()
        
        if self.server_url.startswith("https://"):
            self.ws_url = self.server_url.replace("https://", "wss://") + f"/api/ws/colab?token={token}"
        else:
            self.ws_url = self.server_url.replace("http://", "ws://") + f"/api/ws/colab?token={token}"

    def upload_artifact(self, file_path):
        """Upload generated file to FusionClip /api/storage/upload"""
        upload_url = f"{self.server_url}/api/storage/upload"
        with open(file_path, "rb") as f:
            files = {"file": (os.path.basename(file_path), f, "image/png")}
            res = requests.post(upload_url, files=files, timeout=30)
            if res.status_code == 200:
                return res.json()
            raise RuntimeError(f"Upload failed ({res.status_code}): {res.text}")

    def send_update(self, payload):
        """Send task update via WebSocket if connected, otherwise via HTTP fallback."""
        if self.ws_connected and self.ws:
            try:
                with self._ws_lock:
                    self.ws.send(json.dumps(payload))
                return
            except Exception as e:
                print(f"WebSocket send failed ({e}), falling back to HTTP.")
        
        status = {"task_complete": "COMPLETED", "task_failed": "FAILED"}.get(payload.get("type"), "PROCESSING")
        try:
            requests.post(
                f"{self.server_url}/api/colab/tasks/update?token={self.token}",
                headers={"Authorization": f"Bearer {self.token}"},
                json={
                    "task_id": payload["task_id"],
                    "status": status,
                    "progress": payload.get("percent", 100 if status == "COMPLETED" else 0),
                    "output": payload.get("output"),
                    "error": payload.get("error")
                },
                timeout=10
            )
        except Exception as http_err:
            print(f"HTTP fallback update failed: {http_err}")

    def report_progress(self, task_id, percent):
        self.send_update({"type": "task_progress", "task_id": task_id, "percent": percent, "status": "PROCESSING"})

    def handle_image_generation(self, task_id, params):
        prompt = (params.get("prompt") or "A creative digital artwork") if isinstance(params, dict) else "A creative digital artwork"
        steps = params.get("steps", 2) if isinstance(params, dict) else 2
        self.report_progress(task_id, 25)
        image = sdxl_pipe(prompt=prompt, num_inference_steps=int(steps), guidance_scale=0.0).images[0]
        self.report_progress(task_id, 75)
        out_path = f"/tmp/colab_{task_id}.png"
        image.save(out_path)
        upload_data = self.upload_artifact(out_path)
        return upload_data

    def execute_task(self, task_id, task_type, params):
        with self._gpu_lock:
            try:
                print(f"Executing {task_type} (ID: {task_id})")
                if task_type == "image_generation":
                    upload_res = self.handle_image_generation(task_id, params)
                else:
                    raise ValueError(f"Unsupported task type on Colab: {task_type}")
                
                complete_payload = {
                    "type": "task_complete",
                    "task_id": task_id,
                    "output": {
                        "url": upload_res.get("url"),
                        "filename": upload_res.get("filename"),
                        "path": upload_res.get("path")
                    }
                }
                self.send_update(complete_payload)
                print(f"Completed task {task_id} successfully!")
            except Exception as e:
                print(f"Task {task_id} failed: {e}")
                err_payload = {"type": "task_failed", "task_id": task_id, "error": str(e)}
                self.send_update(err_payload)

    def start(self):
        print(f"Connecting to FusionClip at {self.ws_url}...")
        def on_open(ws):
            self.ws_connected = True
            print("Connected to FusionClip WebSocket bridge!")
        def on_message(ws, msg):
            data = json.loads(msg)
            if data.get("type") == "task_dispatch":
                threading.Thread(target=self.execute_task, args=(data["task_id"], data["task_type"], data.get("parameters", {}))).start()
        def on_close(ws, status, msg):
            self.ws_connected = False
            print("Disconnected from WebSocket bridge.")
        
        while self.is_running:
            try:
                self.ws = websocket.WebSocketApp(self.ws_url, on_open=on_open, on_message=on_message, on_close=on_close)
                self.ws.run_forever()
            except Exception as e:
                print(f"WebSocket error: {e}")
            print("Reconnecting in 5 seconds...")
            time.sleep(5)


In [ ]:
# Step 6: Start Worker
worker = ColabWorker(FUSIONCLIP_SERVER_URL, FUSIONCLIP_SECRET_KEY)
worker.start()
